# BioListen VN — FSC22 Preprocessing Pipeline (`human_head`) [On-Demand Extraction]

Notebook này thực hiện tiền xử lý cho bộ dữ liệu **FSC22 (Forest Sound Classification)** làm dữ liệu huấn luyện chính phục vụ nhánh phân loại mối đe dọa con người (`human_head`).

### Chiến lược tối ưu dung lượng ổ cứng (On-Demand Extraction):
Nhằm tránh việc Google Colab bị đầy bộ nhớ khi giải nén toàn bộ các file audio raw, notebook này sử dụng phương pháp **giải nén theo yêu cầu (On-Demand/Lazy Extraction)**:
- Lập chỉ mục toàn bộ các file trong file zip raw từ Google Drive.
- Đọc metadata CSV trực tiếp từ file zip vào bộ nhớ RAM.
- Duyệt qua từng file âm thanh, giải nén **duy nhất** file đó ra bộ nhớ tạm của Colab, thực hiện tiền xử lý chuyển thành tensor `.pt` 3 kênh màu, sau đó **xóa ngay lập tức** file raw trước khi chuyển sang mẫu tiếp theo.
- Cuối cùng, nén toàn bộ các file `.pt` kết quả thành file `.zip` duy nhất trên Google Drive.

### Quy trình xử lý âm học (ImageNet & EfficientNet-V2-S Compatibility):
1. Chuẩn hóa về **Mono**, resample về **32,000 Hz**.
2. Cắt/Đệm (Slice/Pad) độ dài cố định đúng **5.0 giây** (160,000 samples).
3. Trích xuất **Log-Mel Spectrogram** ($f_{min} = 50$ Hz, $f_{max} = 15,000$ Hz, $n_{mels} = 128$).
4. Tính toán các đặc trưng **Delta** và **Delta-Delta** để tạo thành 3 kênh màu (R = Mel, G = Delta, B = Delta-Delta).
5. Resize song tuyến (bilinear) về kích thước ảnh vuông `(3, 224, 224)`.
6. Lưu trữ các tensor dưới dạng tệp PyTorch `.pt` và xuất tệp metadata CSV tương ứng.

## 1. Kết nối Google Drive & Thiết lập Đường dẫn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import zipfile
import shutil
from tqdm import tqdm

# Cấu hình đường dẫn đầu vào và đầu ra trên Google Drive
drive_raw_zip = '/content/drive/MyDrive/Datasets/BioListenVN/raw_zips/fsc22-v1.zip'
drive_processed_dir = '/content/drive/MyDrive/Datasets/BioListenVN/processed'

# Đường dẫn làm việc cục bộ trên Colab SSD
local_processed_dir = '/content/fsc22_processed'
os.makedirs(drive_processed_dir, exist_ok=True)
os.makedirs(local_processed_dir, exist_ok=True)

if os.path.exists(drive_raw_zip):
    print(f"Tìm thấy raw zip file tại: {drive_raw_zip}")
else:
    print(f"LỖI: Không tìm thấy file tại {drive_raw_zip}. Vui lòng upload dữ liệu lên Drive.")

## 2. Import các Thư viện cần thiết

In [ ]:
import torch
import torchaudio
import torchaudio.transforms as T
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import IPython.display as ipd
import glob

print("PyTorch version:", torch.__version__)
print("Torchaudio version:", torchaudio.__version__)

## 3. Lập chỉ mục Zip file và Đọc Metadata trực tiếp vào bộ nhớ RAM

In [ ]:
if os.path.exists(drive_raw_zip):
    with zipfile.ZipFile(drive_raw_zip, 'r') as z:
        # Quét các file CSV bên trong zip file
        csv_files = [f for f in z.namelist() if f.endswith('.csv')]
        if len(csv_files) > 0:
            with z.open(csv_files[0]) as f:
                df = pd.read_csv(f)
            print(f"Đã đọc file metadata trực tiếp từ zip: {csv_files[0]} (Số dòng: {len(df)})")

            # Lập chỉ mục đường dẫn tất cả file audio trong zip
            print("Đang lập chỉ mục danh sách file audio trong zip...")
            all_files = z.namelist()
            audio_in_zip = [p for p in all_files if p.endswith('.wav')]
            file_to_zip_path = {os.path.basename(p): p for p in audio_in_zip}
            print(f"Đã lập chỉ mục xong {len(audio_in_zip)} file WAV.")
        else:
            print("LỖI: Không tìm thấy file metadata CSV trong file zip!")

## 4. Cấu hình Tham số và Xây dựng Lớp Preprocessor

In [ ]:
# Cấu hình thống nhất theo tài liệu training_workflow.md
AUDIO_CONFIG = {
    "sample_rate": 32000,
    "duration_sec": 5,
    "n_samples": 32000 * 5,
    "n_fft": 2048,
    "hop_length": 512,
    "n_mels": 128,
    "fmin_human": 50,
    "fmax": 15000,
}

class FSC22AudioPreprocessor:
    def __init__(self, config=AUDIO_CONFIG):
        self.target_sr = config["sample_rate"]
        self.target_samples = config["n_samples"]
        self.n_fft = config["n_fft"]
        self.hop_length = config["hop_length"]
        self.n_mels = config["n_mels"]
        self.fmin = config["fmin_human"]
        self.fmax = config["fmax"]

        self.mel_spectrogram = T.MelSpectrogram(
            sample_rate=self.target_sr,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            n_mels=self.n_mels,
            f_min=self.fmin,
            f_max=self.fmax
        )
        self.amplitude_to_db = T.AmplitudeToDB()

    def preprocess(self, file_path):
        # 1. Load file âm thanh
        waveform, sr = torchaudio.load(file_path)

        # 2. Chuẩn hóa về Mono
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # 3. Resample về 32,000 Hz
        if sr != self.target_sr:
            resampler = T.Resample(orig_freq=sr, new_freq=self.target_sr)
            waveform = resampler(waveform)

        # 4. Temporal Alignment (Cắt/Pad đúng 5s)
        num_samples = waveform.shape[1]
        if num_samples > self.target_samples:
            waveform = waveform[:, :self.target_samples]
        elif num_samples < self.target_samples:
            pad_len = self.target_samples - num_samples
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))

        # 5. Trích xuất Mel-spectrogram & Chuyển sang dB scale
        mel_spec = self.mel_spectrogram(waveform)
        mel_spec_db = self.amplitude_to_db(mel_spec)

        # 6. Chuẩn hóa Min-Max về dải [0, 1] cho Spectrogram gốc
        min_val = mel_spec_db.min()
        max_val = mel_spec_db.max()
        if max_val - min_val > 1e-9:
            mel_spec_norm = (mel_spec_db - min_val) / (max_val - min_val)
        else:
            mel_spec_norm = torch.zeros_like(mel_spec_db)
        
        # 7. Resize song tuyến (Bilinear) về ảnh vuông (224, 224)
        mel_spec_unsqueezed = mel_spec_norm.unsqueeze(0)
        mel_spec_resized = torch.nn.functional.interpolate(
            mel_spec_unsqueezed,
            size=(224, 224),
            mode='bilinear',
            align_corners=False
        ).squeeze(0)

        # 8. Tính Delta & Delta-Delta
        delta = torchaudio.functional.compute_deltas(mel_spec_resized)
        delta2 = torchaudio.functional.compute_deltas(delta)

        # Hàm chuẩn hóa từng kênh về dải [0, 1]
        def norm_tensor(t):
            mn, mx = t.min(), t.max()
            if mx - mn > 1e-9:
                return (t - mn) / (mx - mn)
            return torch.zeros_like(t)

        mel_spec_resized_norm = norm_tensor(mel_spec_resized)
        delta_norm = norm_tensor(delta)
        delta2_norm = norm_tensor(delta2)

        # Ghép thành 3 kênh màu RGB
        mel_rgb = torch.cat([mel_spec_resized_norm, delta_norm, delta2_norm], dim=0)

        return waveform, mel_spec_db, mel_rgb

## 5. Chạy Thử nghiệm và Trực quan hóa Đặc trưng 3 kênh

In [ ]:
if 'df' in locals() and len(df) > 0:
    # Lấy 1 mẫu ngẫu nhiên
    test_row = df.sample(1).iloc[0]
    fname = test_row['Dataset File Name']
    print(f"File test: {fname} | Lớp: {test_row['Class Name']}")

    if fname in file_to_zip_path:
        zip_member = file_to_zip_path[fname]
        temp_test_file = '/content/temp_test.wav'
        
        with zipfile.ZipFile(drive_raw_zip, 'r') as z:
            with z.open(zip_member) as source, open(temp_test_file, 'wb') as target:
                shutil.copyfileobj(source, target)
        
        preprocessor = FSC22AudioPreprocessor()
        waveform, spec_db, mel_rgb = preprocessor.preprocess(temp_test_file)
        
        # Xóa file tạm ngay
        if os.path.exists(temp_test_file):
            os.remove(temp_test_file)
            
        # Vẽ đồ thị kiểm định
        fig, ax = plt.subplots(2, 2, figsize=(15, 10))
        
        # 1. Waveform
        ax[0, 0].plot(waveform[0].numpy())
        ax[0, 0].set_title("Normalized Mono Waveform (32kHz, 5s)")
        
        # 2. Channel 0: Mel Spectrogram
        img1 = ax[0, 1].imshow(mel_rgb[0].numpy(), cmap='magma', origin='lower')
        ax[0, 1].set_title("Channel 0: Log-Mel Spectrogram (224x224)")
        fig.colorbar(img1, ax=ax[0, 1])
        
        # 3. Channel 1: Delta
        img2 = ax[1, 0].imshow(mel_rgb[1].numpy(), cmap='viridis', origin='lower')
        ax[1, 0].set_title("Channel 1: Delta Feature (224x224)")
        fig.colorbar(img2, ax=ax[1, 0])
        
        # 4. Channel 2: Delta-Delta
        img3 = ax[1, 1].imshow(mel_rgb[2].numpy(), cmap='coolwarm', origin='lower')
        ax[1, 1].set_title("Channel 2: Delta-Delta Feature (224x224)")
        fig.colorbar(img3, ax=ax[1, 1])
        
        plt.tight_layout()
        plt.show()
        
        print("Tập RGB tensor shape:", mel_rgb.shape)
        print("Max value:", mel_rgb.max().item(), "| Min value:", mel_rgb.min().item())

## 6. Chạy Tiền xử lý Toàn bộ Dữ liệu với On-Demand Extraction

In [ ]:
if 'df' in locals() and 'file_to_zip_path' in locals():
    processed_records = []
    preprocessor = FSC22AudioPreprocessor()

    print("Bắt đầu tiền xử lý toàn bộ FSC22 sử dụng On-Demand Extraction...")
    temp_process_file = '/content/temp_process.wav'

    with zipfile.ZipFile(drive_raw_zip, 'r') as z:
        for idx, row in tqdm(df.iterrows(), total=len(df), desc="Preprocessing FSC22"):
            fname = row['Dataset File Name']

            if fname in file_to_zip_path:
                zip_member = file_to_zip_path[fname]

                # 1. Giải nén tạm đúng 1 file này
                with z.open(zip_member) as source, open(temp_process_file, 'wb') as target:
                    shutil.copyfileobj(source, target)

                # 2. Thực hiện tiền xử lý
                _, _, mel_rgb = preprocessor.preprocess(temp_process_file)

                # 3. Xóa file tạm ngay lập tức
                if os.path.exists(temp_process_file):
                    os.remove(temp_process_file)

                # 4. Lưu tensor PyTorch cục bộ
                out_name = fname.replace('.wav', '.pt')
                dest_path = os.path.join(local_processed_dir, out_name)
                torch.save(mel_rgb, dest_path)

                # Ghi nhận log metadata dynamically
                record = {
                    'processed_pt_filename': out_name
                }
                for col in df.columns:
                    record[col] = row[col]
                processed_records.append(record)

    processed_df = pd.DataFrame(processed_records)
    print(f"Đã tiền xử lý thành công {len(processed_df)} / {len(df)} mẫu.")

## 7. Lưu Metadata mới & Đồng bộ hóa lên Google Drive

In [ ]:
if 'processed_df' in locals() and len(processed_df) > 0:
    # 1. Lưu file csv metadata mới trực tiếp lên Google Drive
    meta_dest = os.path.join(drive_processed_dir, 'fsc22_processed_metadata.csv')
    processed_df.to_csv(meta_dest, index=False)
    print(f"Đã lưu file metadata mới tại: {meta_dest}")

    # 2. Nén các file .pt kết quả thành file zip trực tiếp trên Drive
    zip_dest_name = os.path.join(drive_processed_dir, 'fsc22_processed')
    print(f"Đang nén các tệp processed .pt vào file zip tại Drive...")
    shutil.make_archive(zip_dest_name, 'zip', local_processed_dir)
    print(f"Đã nén và đồng bộ thành công lên Google Drive: {zip_dest_name}.zip")

## 8. Dọn dẹp Thư mục cục bộ

In [ ]:
print("Đang dọn dẹp các thư mục tạm cục bộ...")
if os.path.exists(local_processed_dir):
    shutil.rmtree(local_processed_dir)
print("Đã dọn dẹp sạch sẽ ổ cứng cục bộ!")